In [ ]:
import pandas as pd
import numpy as np


daily_sales = pd.read_csv(r"C:\Users\DELL\Downloads\StockWise_Website_ProductNames\data\stockwise_sales.csv")
daily_sales = daily_sales.dropna(subset=["Date", "Product ID"]).copy()
daily_sales["Date"] = pd.to_datetime(daily_sales["Date"], errors="coerce")
print(daily_sales.head())


             Date Store ID Product ID     Category Region  Inventory Level  \
0        1/1/2024     S003      P0007    Groceries   East              567   
1        1/1/2024     S005      P0003         Toys  North              118   
2        1/1/2024     S003      P0005         Toys   East              284   
3        1/1/2024     S004      P0010    Groceries   West             1278   
4        1/1/2024     S005      P0018     Clothing  North              155   
...           ...      ...        ...          ...    ...              ...   
75995  12/31/2025     S002      P0014  Electronics  South              146   
75996  12/31/2025     S005      P0007    Groceries  North              192   
75997  12/31/2025     S004      P0008    Furniture   West              154   
75998  12/31/2025     S002      P0018     Clothing  South              153   
75999  12/31/2025     S001      P0017         Toys  North               52   

       Units Sold  Units Ordered  Price  Discount Weather Condi

In [ ]:
daily_sales.isnull().sum()


Date                  0
Store ID              1
Product ID            1
Category              1
Region                1
Inventory Level       1
Units Sold            1
Units Ordered         1
Price                 1
Discount              1
Weather Condition     1
Promotion             1
Competitor Pricing    1
Seasonality           1
Epidemic              1
Demand                1
dtype: int64

In [ ]:
daily_sales.dropna()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,1/1/2021,S001,P0001,Electronics,North,195.0,102.0,252.0,72.72,5.0,Snowy,0.0,85.73,Winter,0.0,115.0
1,1/2/2021,S001,P0002,Clothing,North,117.0,117.0,249.0,80.16,15.0,Snowy,1.0,92.02,Winter,0.0,229.0
2,1/3/2021,S001,P0003,Clothing,North,247.0,114.0,612.0,62.94,10.0,Snowy,1.0,60.08,Winter,0.0,157.0
3,1/4/2021,S001,P0004,Electronics,North,139.0,45.0,102.0,87.63,10.0,Snowy,0.0,85.19,Winter,0.0,52.0
4,1/5/2021,S001,P0005,Groceries,North,152.0,65.0,271.0,54.41,0.0,Snowy,0.0,51.63,Winter,0.0,59.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75995,1/26/2229,S005,P0016,Toys,North,233.0,63.0,0.0,29.80,5.0,Snowy,0.0,32.23,Winter,0.0,64.0
75996,1/27/2229,S005,P0017,Toys,North,137.0,115.0,141.0,42.92,5.0,Snowy,0.0,40.73,Winter,0.0,137.0
75997,1/28/2229,S005,P0018,Clothing,North,197.0,44.0,0.0,17.81,10.0,Snowy,0.0,19.41,Winter,0.0,68.0
75998,1/29/2229,S005,P0019,Furniture,North,125.0,58.0,0.0,151.72,0.0,Snowy,0.0,143.71,Winter,0.0,84.0


In [ ]:
daily_sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 76001 entries, 0 to 76000
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                76001 non-null  str    
 1   Store ID            76000 non-null  str    
 2   Product ID          76000 non-null  str    
 3   Category            76000 non-null  str    
 4   Region              76000 non-null  str    
 5   Inventory Level     76000 non-null  float64
 6   Units Sold          76000 non-null  float64
 7   Units Ordered       76000 non-null  float64
 8   Price               76000 non-null  float64
 9   Discount            76000 non-null  float64
 10  Weather Condition   76000 non-null  str    
 11  Promotion           76000 non-null  float64
 12  Competitor Pricing  76000 non-null  float64
 13  Seasonality         76000 non-null  str    
 14  Epidemic            76000 non-null  float64
 15  Demand              76000 non-null  float64
dtypes: float64(9), 

In [ ]:
daily_sales.shape

(76000, 16)

In [ ]:
daily_sales.duplicated()


0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
dtype: bool

In [ ]:

# CALCULATING DEMAND & VARIABILITY STATS

demand_stats = (
    daily_sales.groupby('Product ID')['Units Sold']
    .agg(
        Average_Daily_Sales='mean',
        Total_Units_Sold='sum',
        Sales_Std='std',
        Days_Recorded='count'
    )
    .reset_index()
)



In [ ]:
# Calculate Coefficient of Variation (CV)
# (Higher CV means more variable/unpredictable demand)
demand_stats['Demand_CV'] = (
    demand_stats['Sales_Std'] /
    demand_stats['Average_Daily_Sales'].replace(0, np.nan)
)


In [ ]:
# Classify products by average daily sales volume
high_threshold = demand_stats['Average_Daily_Sales'].quantile(0.75)
low_threshold = demand_stats['Average_Daily_Sales'].quantile(0.25)

demand_stats['Demand_Level'] = np.select(
    [
        demand_stats['Average_Daily_Sales'] >= high_threshold,
        demand_stats['Average_Daily_Sales'] <= low_threshold
    ],
    ['High Demand', 'Low Demand'],
    default='Medium Demand'
)




daily_sales = daily_sales.merge(
    demand_stats[['Product ID', 'Demand_Level']], 
    on='Product ID', 
    how='left'
)

print(daily_sales)
print(daily_sales)




             Date Store ID Product ID     Category Region  Inventory Level  \
0        1/1/2024     S003      P0007    Groceries   East              567   
1        1/1/2024     S005      P0003         Toys  North              118   
2        1/1/2024     S003      P0005         Toys   East              284   
3        1/1/2024     S004      P0010    Groceries   West             1278   
4        1/1/2024     S005      P0018     Clothing  North              155   
...           ...      ...        ...          ...    ...              ...   
75995  12/31/2025     S002      P0014  Electronics  South              146   
75996  12/31/2025     S005      P0007    Groceries  North              192   
75997  12/31/2025     S004      P0008    Furniture   West              154   
75998  12/31/2025     S002      P0018     Clothing  South              153   
75999  12/31/2025     S001      P0017         Toys  North               52   

       Units Sold  Units Ordered  Price  Discount Weather Condi

In [ ]:
# Identify irregular demand using CV threshold (> 0.5)
demand_stats['Demand_Pattern'] = np.where(
    demand_stats['Demand_CV'] > 0.5,
    'Irregular Demand',
    'Relatively Stable Demand'
)


In [ ]:

#  DISPLAY RESULTS

print("\n--- TOP 10 PRODUCTS BY AVERAGE DAILY SALES ---")
print(
    demand_stats.sort_values(
        'Average_Daily_Sales', ascending=False
    ).head(10)
)




--- TOP 10 PRODUCTS BY AVERAGE DAILY SALES ---
   Product ID  Average_Daily_Sales  Total_Units_Sold  Sales_Std  \
6       P0007            99.923421            379709  48.710524   
3       P0004            99.375526            377627  46.665029   
8       P0009            99.150263            376771  42.822255   
12      P0013            99.136053            376717  49.043010   
1       P0002            94.567105            359355  45.808478   
9       P0010            94.026053            357299  44.774644   
0       P0001            93.898421            356814  46.670477   
4       P0005            92.569737            351765  43.501905   
5       P0006            90.681316            344589  44.902729   
2       P0003            89.876579            341531  43.706044   

    Days_Recorded  Demand_CV   Demand_Level            Demand_Pattern  
6            3800   0.487479    High Demand  Relatively Stable Demand  
3            3800   0.469583    High Demand  Relatively Stable Demand 

In [ ]:
print("\n--- TOP 10 PRODUCTS BY DEMAND VARIABILITY ---")
print(
    demand_stats.sort_values(
        'Demand_CV', ascending=False
    )[['Product ID', 'Average_Daily_Sales', 'Sales_Std', 'Demand_CV', 'Demand_Pattern']].head(10)
)


--- TOP 10 PRODUCTS BY DEMAND VARIABILITY ---
   Product ID  Average_Daily_Sales  Sales_Std  Demand_CV  \
15      P0016            85.978421  46.039491   0.535477   
13      P0014            83.525526  43.791706   0.524291   
14      P0015            88.274474  45.759091   0.518373   
19      P0020            73.147632  37.896112   0.518077   
16      P0017            77.610000  38.635690   0.497818   
0       P0001            93.898421  46.670477   0.497032   
5       P0006            90.681316  44.902729   0.495171   
12      P0013            99.136053  49.043010   0.494704   
11      P0012            77.962105  38.458178   0.493293   
10      P0011            87.600789  43.048114   0.491412   

              Demand_Pattern  
15          Irregular Demand  
13          Irregular Demand  
14          Irregular Demand  
19          Irregular Demand  
16  Relatively Stable Demand  
0   Relatively Stable Demand  
5   Relatively Stable Demand  
12  Relatively Stable Demand  
11  Relativel

In [ ]:
#Predicting demand

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import pandas as pd
import numpy as np
from pathlib import Path
import json


daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])
daily_sales['Year'] = daily_sales['Date'].dt.year
daily_sales['Month'] = daily_sales['Date'].dt.month
daily_sales['Day'] = daily_sales['Date'].dt.day
daily_sales['DayOfWeek'] = daily_sales['Date'].dt.dayofweek

categorical_cols = ['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']
label_encoders = {}


for col in categorical_cols:
    le = LabelEncoder()
    daily_sales[col] = le.fit_transform(daily_sales[col].astype(str))
    label_encoders[col] = le

feature_columns = ['Product ID', 'Inventory Level', 'Units Ordered', 'Units Sold', 'Price', 'Discount', 'Year', 'Month', 'Day', 'Competitor Pricing', 'Seasonality']
x = daily_sales[feature_columns]
y = daily_sales['Demand']


X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, objective='reg:squarederror')

print("Training the XGBoost model...")
model.fit(X_train, y_train)

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)
model_path = models_dir / "xgboost_stockwise_model.pkl"
import pickle
with open(model_path, "wb") as f:
    pickle.dump(model, f)

meta = {
    "Product ID": sorted(daily_sales["Product ID"].unique().tolist()),
    "Seasonality": sorted(daily_sales["Seasonality"].unique().tolist()),
}
with open(models_dir / "encoding_classes.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print("Model and metadata saved successfully!")


In [ ]:
print("Model saved successfully using joblib!")


Training the XGBoost model...
Model saved successfully using joblib!


In [ ]:
#Evaluate the Model
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n--- Model Evaluation ---")
print(f"MAE (Mean Absolute Error): {mae:.2f}")
print(f"RMSE (Root Mean Squared Error): {rmse:.2f}")
print(f"R2 Score: {r2:.2f}")



--- Model Evaluation ---
MAE (Mean Absolute Error): 13.75
RMSE (Root Mean Squared Error): 18.27
R² Score: 0.85


In [ ]:
#Predicting when the stock will finish and when should i reorder 

import pandas as pd
import numpy as np

def calculate_stock_schedule(model, X_live_row, current_inventory, lead_time_days=3, safety_stock_buffer=20):
    """
    Predicts daily demand using XGBoost and computes stockout timeline and reorder triggers.
    """
    # 1. Predict daily demand for the specific product/store using the XGBoost model
    predicted_daily_demand = model.predict(X_live_row)[0]
    
    # Handle edge case where demand is zero or negative
    if predicted_daily_demand <= 0:
        return {
            "Predicted Daily Demand": 0,
            "Days Until Stockout": "Infinite (No sales predicted)",
            "Reorder Status": "NO ACTION NEEDED "
        }


    
    # 2. Calculate when stock finishes
    days_to_stockout = current_inventory / predicted_daily_demand


    
    # 3. Calculate Reorder Point (When you must place a new order)
    reorder_point = (predicted_daily_demand * lead_time_days) + safety_stock_buffer


    
    # 4. Check if inventory has breached the reorder threshold right now
    is_urgent_reorder = current_inventory <= reorder_point
    
    return {
        "Predicted Daily Demand": round(predicted_daily_demand, 2),
        "Days Until Stockout": round(days_to_stockout, 1),
        "Reorder Point Threshold": round(reorder_point, 1),
        "Current Inventory": current_inventory,
        "Reorder Status": "REORDER NOW 🚨" if is_urgent_reorder else "STOCK LEVEL SAFE ✅"
    }



# --- Example of how to run it for a product from your test set ---
# (Assuming X_test and model are already defined from your training script)
sample_index = 8
sample_features = X_test.iloc[[sample_index]]
sample_inventory_level = 500  # Example current stock from your columns

schedule_result = calculate_stock_schedule(
    model=model, 
    X_live_row=sample_features, 
    current_inventory=sample_inventory_level, 
    lead_time_days=8,         # Supplier takes 4 days to deliver
    safety_stock_buffer=50   # Buffer stock
)




# Generate the report
schedule_result = calculate_stock_schedule(
    model=model,
    X_live_row=sample_features,
    current_inventory=sample_inventory_level,
    lead_time_days=8,
    safety_stock_buffer=50
)

# Convert results into a table
report_df = pd.DataFrame(
    list(schedule_result.items()),
    columns=["Inventory Metrics", "Results"]
)

# Display report title
print("\n" + "=" * 55)
print("       STOCKWISE - INVENTORY & REORDER REPORT")
print("=" * 55)

# Display table
print(report_df.to_string(index=False))

print("=" * 55)


       STOCKWISE - INVENTORY & REORDER REPORT
      Inventory Metrics            Results
 Predicted Daily Demand          25.129999
    Days Until Stockout               19.9
Reorder Point Threshold              251.0
      Current Inventory                500
         Reorder Status STOCK LEVEL SAFE ✅
